In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# EXP_LOSS — Loss Function Ablation Study

본 실험은 파고 예측 모델의 피크 과소추정(Under-prediction)을 완화하고, 대회 핵심 지표인 `comp_rmse`($h_s \ge 1.5\text{m}$)를 극대화하기 위한 **손실 함수 비교 실험**입니다.

### 검증 후보군 (4가지)
1. **`1_TargetAware_Weighted`**: 기준 시점이 아니라 예측 대상 시점(타깃 $y$)이 $1.5\text{m}$ 이상일 때 가중치(2.0 vs 0.2) 집중 부여
2. **`2_Continuous_Smooth`**: Hard threshold 경계면 불안정을 방지하기 위해 타깃 크기에 비례($1.0 + 0.5y$)하는 연속 가중치 부여
3. **`3_Asymmetric_Penalty`**: 실제값보다 낮게 예측한 경우(Under-prediction) 오차를 $1.6$배 페널티로 증폭
4. **`4_Pure_RMSE`**: 가중치 없는 표준 베이스라인 RMSE 손실 함수

In [ ]:
# ============================================================
# 0. SETUP & PACKAGES
# ============================================================
from pathlib import Path
import gc
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


In [ ]:
# ============================================================
# 1. CONFIG & DATA PREPARATION
# ============================================================
DATA_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v6.csv")
TIME_COL = "time"
STATION_COL = "station"
STEP_MINUTES = 10
STEPS_PER_HOUR = 6

INPUT_LEN = 289
LEAD_HOURS = [3, 6, 9, 12, 18, 24]
LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]
MAX_LEAD = max(LEAD_STEPS)
N_TARGETS = len(LEAD_STEPS)

df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

if "hs_original_observed" not in df.columns:
    df["hs_original_observed"] = df["hs"].notna().astype(np.int8)

FEATURES = [
    "hs", "tp", "hmax", "wspd", "gust", "u_wind", "v_wind",
    "wspd_mean_6h", "wspd_mean_12h", "gust_max_6h", "gust_max_12h",
    "gust_minus_wspd", "caph_change_3h", "caph_change_6h", "caph_change_12h",
    "wave_steepness", "wave_energy", "effective_wind_forcing", "u_wave", "v_wave"
]
FEATURES = [c for c in FEATURES if c in df.columns]

STATION_TO_ID = {st: i for i, st in enumerate(sorted(df[STATION_COL].unique()))}
WINDOW_META_KEYS = ("station_id", "start_idx", "end_idx", "origin_hs", "origin_time")

def build_samples(frame, features, input_len=INPUT_LEN, lead_steps=LEAD_STEPS):
    station_id, start_idx, end_idx, origin_hs, origin_time = [], [], [], [], []
    x_by_station, hs_by_station = {}, {}
    for station, g in frame.groupby(STATION_COL, sort=False):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        sid = STATION_TO_ID[station]
        Xv = g.loc[:, features].to_numpy(dtype=np.float32, copy=True)
        hsv = g["hs"].to_numpy(dtype=np.float32, copy=True)
        obs = g["hs_original_observed"].to_numpy(dtype=np.int8, copy=True)
        times = g[TIME_COL].to_numpy(copy=True)
        x_by_station[sid] = Xv
        hs_by_station[sid] = hsv
        dt = pd.Series(g[TIME_COL]).diff().dt.total_seconds().div(60).to_numpy()
        for e in range(input_len - 1, len(g) - max_lead):
            s = e - input_len + 1
            if not np.all(dt[s + 1:e + 1] == STEP_MINUTES): continue
            if not np.isfinite(Xv[s:e + 1]).all(): continue
            target_idx = np.asarray([e + step for step in lead_steps], dtype=np.int64)
            if not np.isfinite(hsv[target_idx]).all() or not np.all(obs[target_idx] == 1): continue
            if not np.isfinite(hsv[e]): continue
            station_id.append(sid); start_idx.append(s); end_idx.append(e)
            origin_hs.append(hsv[e]); origin_time.append(times[e])
    return {
        "station_id": np.asarray(station_id, dtype=np.int64),
        "start_idx": np.asarray(start_idx, dtype=np.int64),
        "end_idx": np.asarray(end_idx, dtype=np.int64),
        "origin_hs": np.asarray(origin_hs, dtype=np.float32),
        "origin_time": np.asarray(origin_time, dtype="datetime64[ns]"),
        "X_by_station": x_by_station, "hs_by_station": hs_by_station,
        "source_frame": frame, "features": list(features), "n_features": len(features),
    }

def chronological_split(samples, train_ratio=0.80):
    times = pd.Series(pd.to_datetime(samples["origin_time"]))
    source_tz = samples["source_frame"][TIME_COL].dt.tz
    current_tz = times.dt.tz
    if source_tz is not None:
        times = times.dt.tz_localize(source_tz) if current_tz is None else times.dt.tz_convert(source_tz)
    elif current_tz is not None:
        times = times.dt.tz_localize(None)
    cutoff = times.quantile(train_ratio)
    train_mask = (times <= cutoff).to_numpy()
    valid_mask = (times > cutoff).to_numpy()
    def subset(mask):
        part = {key: samples[key][mask] for key in WINDOW_META_KEYS}
        part.update({
            "X_by_station": samples["X_by_station"], "hs_by_station": samples["hs_by_station"],
            "source_frame": samples["source_frame"], "features": samples["features"],
            "n_features": samples["n_features"], "split_cutoff": cutoff,
        })
        return part
    return subset(train_mask), subset(valid_mask), cutoff

def scale_samples(train, valid):
    scaler = StandardScaler()
    train_rows = train["source_frame"][TIME_COL] <= train["split_cutoff"]
    scaler.fit(train["source_frame"].loc[train_rows, train["features"]])
    for Xv in train["X_by_station"].values():
        scaler.transform(Xv, copy=False)
    return dict(train), dict(valid), scaler

class WaveDataset(Dataset):
    def __init__(self, samples):
        self.X_by_station = samples["X_by_station"]
        self.hs_by_station = samples["hs_by_station"]
        self.station_id = samples["station_id"]
        self.start_idx = samples["start_idx"]
        self.end_idx = samples["end_idx"]
        self.origin_hs = samples["origin_hs"]
        self.lead_steps = np.asarray(LEAD_STEPS, dtype=np.int64)
    def __len__(self): return len(self.end_idx)
    def __getitem__(self, idx):
        sid = int(self.station_id[idx])
        start, end = int(self.start_idx[idx]), int(self.end_idx[idx])
        X = torch.from_numpy(self.X_by_station[sid][start:end + 1])
        y = torch.from_numpy(self.hs_by_station[sid][end + self.lead_steps])
        return X, y, torch.tensor(self.origin_hs[idx], dtype=torch.float32), torch.tensor(sid)

def make_loader(samples, batch_size=128, shuffle=False):
    ds = WaveDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=False, drop_last=False)

raw_samples = build_samples(df, FEATURES)
train_samples, valid_samples, _ = chronological_split(raw_samples)
train_samples, valid_samples, _ = scale_samples(train_samples, valid_samples)
print("Data Ready -> Train:", len(train_samples["end_idx"]), "| Valid:", len(valid_samples["end_idx"]))


In [ ]:
# ============================================================
# 2. MODEL & 4 LOSS CANDIDATES DEFINITIONS
# ============================================================
class ITransformer(nn.Module):
    def __init__(self, seq_len, n_features, pred_len, d_model=64, n_heads=2, e_layers=4, dropout=0.25):
        super().__init__()
        self.value_embedding = nn.Linear(seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, pred_len)
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.value_embedding(x)
        x = self.encoder(x)
        x = self.norm(x)
        return self.head(x)

# --- 4가지 Loss 후보군 ---
class Loss_TargetAware(nn.Module):
    """후보 1: 타깃 y 자체가 1.5m 이상일 때 가중치 집중"""
    def __init__(self, threshold=1.5, high_w=2.0, low_w=0.3):
        super().__init__()
        self.threshold = threshold; self.high_w = high_w; self.low_w = low_w
    def forward(self, pred, target, origin_hs):
        diff_sq = (pred - target) ** 2
        weights = torch.where(target >= self.threshold, self.high_w, self.low_w)
        return torch.sqrt(torch.mean(weights * diff_sq) + 1e-6)

class Loss_ContinuousSmooth(nn.Module):
    """후보 2: 타깃 파고에 비례하는 부드러운 연속 가중치"""
    def forward(self, pred, target, origin_hs):
        diff_sq = (pred - target) ** 2
        weights = 1.0 + 0.5 * torch.clamp(target, min=0.0, max=5.0)
        return torch.sqrt(torch.mean(weights * diff_sq) + 1e-6)

class Loss_Asymmetric(nn.Module):
    """후보 3: 과소추정(Under-prediction) 오차 집중 페널티"""
    def __init__(self, penalty=1.6):
        super().__init__()
        self.penalty = penalty
    def forward(self, pred, target, origin_hs):
        diff = pred - target
        weights = torch.where(diff < 0, self.penalty, 1.0)
        return torch.sqrt(torch.mean(weights * (diff ** 2)) + 1e-6)

class Loss_PureRMSE(nn.Module):
    """후보 4: 표준 무가중 RMSE"""
    def forward(self, pred, target, origin_hs):
        return torch.sqrt(torch.mean((pred - target) ** 2) + 1e-6)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def competition_like_rmse(y_true, y_pred, origin_hs):
    mask = origin_hs >= 1.5
    if mask.sum() == 0: return np.nan, 0
    return rmse(y_true[mask], y_pred[mask]), int(mask.sum())

def evaluate_model(model, loader):
    model.eval()
    preds, ys, origins = [], [], []
    with torch.no_grad():
        for X, y, origin_hs, _ in loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            preds.append(model(X).cpu().numpy())
            ys.append(y.cpu().numpy())
            origins.append(origin_hs.numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    origin_hs = np.concatenate(origins)
    comp, comp_n = competition_like_rmse(y_true, y_pred, origin_hs)
    return {"overall_rmse": rmse(y_true, y_pred), "comp_rmse": comp, "comp_valid_n": comp_n}


In [ ]:
# ============================================================
# 3. RUN LOSS ABLATION (COMPARING 4 LOSSES)
# ============================================================
LOSS_CANDIDATES = {
    "1_TargetAware": Loss_TargetAware(threshold=1.5, high_w=2.0, low_w=0.3),
    "2_ContinuousSmooth": Loss_ContinuousSmooth(),
    "3_Asymmetric": Loss_Asymmetric(penalty=1.6),
    "4_PureRMSE": Loss_PureRMSE(),
}

FIXED_PARAMS = {
    "d_model": 64, "n_heads": 2, "e_layers": 4, "dropout": 0.25,
    "lr": 2.0e-5, "weight_decay": 0.0006, "batch_size": 128
}

EPOCHS_PER_LOSS = 10
results = []
epoch_histories = {}

print("=" * 75)
print("STARTING LOSS ABLATION (10 EPOCHS PER CANDIDATE)")
print("=" * 75)

for name, criterion in LOSS_CANDIDATES.items():
    print(f"\n>>> Testing: {name} ...")
    seed_everything(SEED)
    train_loader = make_loader(train_samples, batch_size=FIXED_PARAMS["batch_size"], shuffle=True)
    valid_loader = make_loader(valid_samples, batch_size=FIXED_PARAMS["batch_size"], shuffle=False)

    model = ITransformer(
        seq_len=INPUT_LEN, n_features=train_samples["n_features"], pred_len=N_TARGETS,
        d_model=FIXED_PARAMS["d_model"], n_heads=FIXED_PARAMS["n_heads"],
        e_layers=FIXED_PARAMS["e_layers"], dropout=FIXED_PARAMS["dropout"]
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=FIXED_PARAMS["lr"], weight_decay=FIXED_PARAMS["weight_decay"])

    best_comp = np.inf
    best_epoch = 0
    best_overall = np.inf
    history = []

    for ep in range(1, EPOCHS_PER_LOSS + 1):
        model.train()
        losses = []
        for X, y, origin_hs, _ in train_loader:
            X, y, origin_hs = X.to(DEVICE), y.to(DEVICE), origin_hs.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X), y, origin_hs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        m = evaluate_model(model, valid_loader)
        history.append({"epoch": ep, "train_loss": np.mean(losses), **m})
        print(f"Ep {ep:02d} | train={np.mean(losses):.4f} | overall={m['overall_rmse']:.4f} | comp_rmse={m['comp_rmse']:.4f}")

        if m["comp_rmse"] < best_comp:
            best_comp = m["comp_rmse"]
            best_epoch = ep
            best_overall = m["overall_rmse"]

    results.append({
        "Loss_Name": name,
        "Best_comp_rmse": best_comp,
        "Best_Epoch": best_epoch,
        "Overall_at_Best": best_overall
    })
    epoch_histories[name] = pd.DataFrame(history)
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# ============================================================
# 4. COMPARISON RESULTS & PLOT
# ============================================================
res_df = pd.DataFrame(results).sort_values("Best_comp_rmse").reset_index(drop=True)
print("\n" + "=" * 75)
print("★ FINAL LOSS ABLATION LEADERBOARD")
print("=" * 75)
display(res_df)

plt.figure(figsize=(9, 5))
for name, hdf in epoch_histories.items():
    plt.plot(hdf["epoch"], hdf["comp_rmse"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation comp_rmse (m)")
plt.title("Loss Candidates Validation comp_rmse Trajectory")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
